# SageMaker Fraud Detection Pipeline - Execution Notebook

This notebook allows you to:
1. Create/update the SageMaker pipeline
2. Start pipeline executions
3. Monitor execution progress
4. View results and metrics
5. Manage pipeline versions

**Environment:** Run this in a SageMaker AI Notebook instance

## 1. Setup and Configuration

In [1]:
# Install project in editable mode so changes to source code are immediately available
! uv pip install -e ../

Using Python 3.12.2 environment at: /Users/skoppar/workspace/sample-mlops-bestpractices/.venv
Resolved 208 packages in 1.05s                                       
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-automated-drift-and-trend-monitoring @ file:///Users/skopp
   Building sagemaker-auto

In [2]:
import sys
print(sys.executable)


/Users/skoppar/workspace/sample-mlops-bestpractices/.venv/bin/python


In [3]:
from importlib.metadata import version
print(f"SageMaker SDK version: {version('sagemaker')}")

SageMaker SDK version: 3.7.1


## 2.4. Required IAM Permissions

**⚠️ ACTION REQUIRED:** Your SageMaker role needs additional permissions for the full pipeline to work.

The role **cannot modify itself** from within this notebook (AWS security best practice). You need to add these permissions manually.

### Required Permissions

Copy this JSON and add it as an **inline policy** to your SageMaker role:

**Role ARN:** `arn:aws:iam::YOUR_ACCOUNT_ID:role/service-role/AmazonSageMaker-ExecutionRole-20250722T131288`

**Policy Name:** `FraudDetectionPipelinePermissions`

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "GlueDataCatalogAccess",
      "Effect": "Allow",
      "Action": [
        "glue:GetDatabase",
        "glue:GetDatabases",
        "glue:GetTable",
        "glue:GetTables",
        "glue:GetPartition",
        "glue:GetPartitions",
        "glue:BatchGetPartition",
        "glue:GetCatalogImportStatus"
      ],
      "Resource": "*"
    },
    {
      "Sid": "LakeFormationDataAccess",
      "Effect": "Allow",
      "Action": [
        "lakeformation:GetDataAccess",
        "lakeformation:GetResourceLFTags",
        "lakeformation:ListLFTags",
        "lakeformation:GetLFTag",
        "lakeformation:SearchTablesByLFTags",
        "lakeformation:SearchDatabasesByLFTags"
      ],
      "Resource": "*"
    },
    {
      "Sid": "SQSManagement",
      "Effect": "Allow",
      "Action": [
        "sqs:CreateQueue",
        "sqs:DeleteQueue",
        "sqs:GetQueueUrl",
        "sqs:GetQueueAttributes",
        "sqs:SetQueueAttributes",
        "sqs:SendMessage",
        "sqs:ReceiveMessage",
        "sqs:DeleteMessage",
        "sqs:ListQueues"
      ],
      "Resource": "*"
    },
    {
      "Sid": "LambdaManagement",
      "Effect": "Allow",
      "Action": [
        "lambda:CreateFunction",
        "lambda:UpdateFunctionCode",
        "lambda:UpdateFunctionConfiguration",
        "lambda:GetFunction",
        "lambda:ListFunctions",
        "lambda:InvokeFunction",
        "lambda:CreateEventSourceMapping",
        "lambda:DeleteEventSourceMapping",
        "lambda:GetEventSourceMapping",
        "lambda:UpdateEventSourceMapping"
      ],
      "Resource": "*"
    },
    {
      "Sid": "IAMPassRole",
      "Effect": "Allow",
      "Action": "iam:PassRole",
      "Resource": "arn:aws:iam::YOUR_ACCOUNT_ID:role/service-role/AmazonSageMaker-ExecutionRole-20250722T131288",
      "Condition": {
        "StringEquals": {
          "iam:PassedToService": [
            "sagemaker.amazonaws.com",
            "lambda.amazonaws.com"
          ]
        }
      }
    }
  ]
}
```

### How to Add (AWS Console)

1. Go to [IAM Console → Roles](https://console.aws.amazon.com/iam/home#/roles)
2. Search for: `AmazonSageMaker-ExecutionRole-20250722T131288`
3. Click the role name
4. Click **Add permissions** → **Create inline policy**
5. Click **JSON** tab
6. Paste the policy above
7. Click **Review policy**
8. Name it: `FraudDetectionPipelinePermissions`
9. Click **Create policy**

### What These Permissions Enable

- **Glue Data Catalog:** PySpark reads Athena tables via Glue
- **Lake Formation:** Data access control for Athena tables
- **SQS:** Inference logging queue (optional)
- **Lambda:** Inference logging function (optional)
- **IAM PassRole:** Allows SageMaker to pass role to Lambda

### Alternative: Lake Formation Console

If you're using Lake Formation, also grant data permissions:
1. Go to [Lake Formation Console](https://console.aws.amazon.com/lakeformation/home)
2. Navigate to **Permissions** → **Data lake permissions**
3. Click **Grant**
4. Select IAM role: `AmazonSageMaker-ExecutionRole-20250722T131288`
5. Database: `fraud_detection`
6. Tables: `training_data`, `inference_responses`, `ground_truth`
7. Permissions: `SELECT`, `DESCRIBE`
8. Click **Grant**

## 2.5. Setup SQS Lambda Infrastructure (Optional)

Configure SQS queue and Lambda consumer for real-time inference logging to Athena.

**This cell will:**
1. Check if the infrastructure already exists
2. If not, show you the IAM policy needed to create it
3. Provide options to proceed

**Note:** Inference logging is optional. The pipeline will work without it, but predictions won't be automatically logged to Athena for monitoring.

In [4]:
# Setup Python path for imports
import sys
from pathlib import Path

# Determine project root (2 levels up from this notebook)
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir

# Add project root to path for imports
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Now import required modules
import boto3
from botocore.exceptions import ClientError

print("=" * 80)
print("Checking SQS Lambda Infrastructure for Inference Logging")
print("=" * 80)

# Import config after path is set
from src.config.config import SQS_QUEUE_NAME, LAMBDA_LOGGER_NAME, AWS_DEFAULT_REGION

# Initialize clients
sqs_client = boto3.client('sqs', region_name=AWS_DEFAULT_REGION)
lambda_client = boto3.client('lambda', region_name=AWS_DEFAULT_REGION)
account_id = boto3.client('sts', region_name=AWS_DEFAULT_REGION).get_caller_identity()['Account']

# Check if infrastructure already exists
queue_exists = False
lambda_exists = False

print("\nChecking existing infrastructure...\n")

# Check SQS Queue
try:
    response = sqs_client.get_queue_url(QueueName=SQS_QUEUE_NAME)
    queue_url = response['QueueUrl']
    print(f"✓ SQS Queue already exists: {queue_url}")
    queue_exists = True
except ClientError as e:
    if e.response['Error']['Code'] == 'AWS.SimpleQueueService.NonExistentQueue':
        print(f"○ SQS Queue does not exist: {SQS_QUEUE_NAME}")
    else:
        print(f"? Error checking queue: {e}")

# Check Lambda Function
try:
    response = lambda_client.get_function(FunctionName=LAMBDA_LOGGER_NAME)
    lambda_arn = response['Configuration']['FunctionArn']
    print(f"✓ Lambda function already exists: {lambda_arn}")
    lambda_exists = True
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print(f"○ Lambda function does not exist: {LAMBDA_LOGGER_NAME}")
    else:
        print(f"? Error checking Lambda: {e}")

print("\n" + "=" * 80)

# Decide what to do
if queue_exists and lambda_exists:
    print("✓ INFRASTRUCTURE ALREADY EXISTS - NO SETUP NEEDED!")
    print("=" * 80)
    print("\nYou can skip to the next section. The inference logging infrastructure is ready.")
    SKIP_SQS_SETUP = True
    
elif not queue_exists or not lambda_exists:
    print("⚠ INFRASTRUCTURE SETUP REQUIRED")
    print("=" * 80)
    
    missing = []
    if not queue_exists:
        missing.append("SQS Queue")
    if not lambda_exists:
        missing.append("Lambda Function")
    
    print(f"\nMissing components: {', '.join(missing)}")
    print("\nTo create these, your SageMaker role needs additional permissions.")
    print("\n" + "-" * 80)
    print("OPTION 1: Ask your AWS Administrator to add this inline policy")
    print("-" * 80)
    print(f"\nRole ARN: arn:aws:iam::{account_id}:role/service-role/AmazonSageMaker-ExecutionRole-20250722T131288")
    print("\nPolicy Name: SQSLambdaInfrastructurePermissions")
    print("\nPolicy JSON:")
    print("""
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "SQSManagement",
      "Effect": "Allow",
      "Action": [
        "sqs:CreateQueue",
        "sqs:DeleteQueue",
        "sqs:GetQueueUrl",
        "sqs:GetQueueAttributes",
        "sqs:SetQueueAttributes",
        "sqs:SendMessage",
        "sqs:ReceiveMessage",
        "sqs:DeleteMessage",
        "sqs:ListQueues"
      ],
      "Resource": "*"
    },
    {
      "Sid": "LambdaManagement",
      "Effect": "Allow",
      "Action": [
        "lambda:CreateFunction",
        "lambda:UpdateFunctionCode",
        "lambda:UpdateFunctionConfiguration",
        "lambda:GetFunction",
        "lambda:ListFunctions",
        "lambda:InvokeFunction",
        "lambda:CreateEventSourceMapping",
        "lambda:DeleteEventSourceMapping",
        "lambda:GetEventSourceMapping",
        "lambda:UpdateEventSourceMapping"
      ],
      "Resource": "*"
    },
    {
      "Sid": "IAMPassRole",
      "Effect": "Allow",
      "Action": "iam:PassRole",
      "Resource": "arn:aws:iam::%s:role/service-role/AmazonSageMaker-ExecutionRole-20250722T131288",
      "Condition": {
        "StringEquals": {
          "iam:PassedToService": "lambda.amazonaws.com"
        }
      }
    }
  ]
}
    """ % account_id)
    
    print("\n" + "-" * 80)
    print("OPTION 2: Run setup from AWS CLI with admin credentials")
    print("-" * 80)
    print("\nFrom your local machine with AWS admin credentials:")
    print(f"  python src/setup/setup_inference_logging.py")
    
    print("\n" + "-" * 80)
    print("OPTION 3: Continue without inference logging")
    print("-" * 80)
    print("\nYou can skip this infrastructure and run the pipeline without")
    print("real-time inference logging to Athena. Model inference will still work,")
    print("but predictions won't be automatically logged to the Athena table.")
    
    SKIP_SQS_SETUP = True

print("\n" + "=" * 80)

Checking SQS Lambda Infrastructure for Inference Logging

Checking existing infrastructure...

✓ SQS Queue already exists: https://sqs.us-east-1.amazonaws.com/<ACCOUNT_ID>/fraud-inference-logs
✓ Lambda function already exists: arn:aws:lambda:us-east-1:<ACCOUNT_ID>:function:fraud-inference-log-consumer

✓ INFRASTRUCTURE ALREADY EXISTS - NO SETUP NEEDED!

You can skip to the next section. The inference logging infrastructure is ready.



In [5]:
# Setup SQS Lambda Infrastructure for Inference Logging
# Creates SQS queue + Lambda consumer, then updates .env with the queue URL

import sys
import os
from pathlib import Path
import importlib
from dotenv import load_dotenv

# Setup Python path for imports
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load .env file explicitly
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(env_path, override=True)
    print(f"✓ Loaded environment from: {env_path}")
else:
    print(f"⚠ .env file not found at: {env_path}")

import boto3
from botocore.exceptions import ClientError

print("=" * 80)
print("Setting up SQS Lambda Infrastructure for Inference Logging")
print("=" * 80)

# Import and reload to get latest version after fixes
import src.setup.setup_inference_logging as setup_module
importlib.reload(setup_module)
from src.setup.setup_inference_logging import setup_sqs_queue, setup_lambda_consumer
from src.config.config import SQS_QUEUE_NAME, AWS_DEFAULT_REGION

# Get role: Priority 1 = .env, Priority 2 = get_execution_role()
role = os.getenv('SAGEMAKER_EXEC_ROLE')

if not role:
    print("\n⚠ SAGEMAKER_EXEC_ROLE not found in .env, trying get_execution_role()...")
    from sagemaker.core.helper.session_helper import get_execution_role
    try:
        role = get_execution_role()
        print(f"✓ Using role from SageMaker session")
    except Exception as e:
        raise ValueError(
            "Could not determine SageMaker execution role.\n"
            "Please add SAGEMAKER_EXEC_ROLE to .env file or ensure you're running in a SageMaker notebook."
        )
else:
    print(f"✓ Using role from .env")

# Get AWS account ID
account_id = boto3.client('sts', region_name=AWS_DEFAULT_REGION).get_caller_identity()['Account']

print(f"\nConfiguration:")
print(f"  Queue Name: {SQS_QUEUE_NAME}")
print(f"  Region: {AWS_DEFAULT_REGION}")
print(f"  Role: {role}")
print()

try:
    # Step 1: Setup SQS queue
    print("Step 1: Creating SQS Queue...")
    queue_url = setup_sqs_queue()
    print(f"✓ SQS Queue created: {queue_url}")
    
    # Step 2: Update .env with SQS queue URL so pipeline picks it up
    print("\nStep 2: Updating .env with SQS_QUEUE_URL...")
    
    env_content = env_path.read_text() if env_path.exists() else ""
    
    # Update or add SQS_QUEUE_URL
    if 'SQS_QUEUE_URL=' in env_content:
        # Replace existing line
        lines = env_content.split('\n')
        lines = [f'SQS_QUEUE_URL={queue_url}' if line.startswith('SQS_QUEUE_URL=') else line for line in lines]
        env_content = '\n'.join(lines)
    else:
        # Append SQS config section
        if not env_content.endswith('\n'):
            env_content += '\n'
        env_content += f'\n# SQS Queue for inference logging (auto-configured)\n'
        env_content += f'SQS_QUEUE_URL={queue_url}\n'
    
    # Update or add SQS_QUEUE_NAME
    if 'SQS_QUEUE_NAME=' not in env_content:
        env_content += f'SQS_QUEUE_NAME={SQS_QUEUE_NAME}\n'
    
    # Update or add LAMBDA_LOGGER_NAME
    if 'LAMBDA_LOGGER_NAME=' not in env_content:
        env_content += f'LAMBDA_LOGGER_NAME=fraud-inference-log-consumer\n'
    
    env_path.write_text(env_content)
    
    # Reload .env so the rest of the notebook picks it up
    load_dotenv(env_path, override=True)
    
    print(f"✓ .env updated with SQS_QUEUE_URL={queue_url}")
    
    # Step 3: Setup Lambda consumer
    print("\nStep 3: Creating Lambda function...")
    queue_arn = f"arn:aws:sqs:{AWS_DEFAULT_REGION}:{account_id}:{SQS_QUEUE_NAME}"
    lambda_arn = setup_lambda_consumer(role, queue_arn)
    print(f"✓ Lambda consumer created")
    
    if lambda_arn:
        print(f"  Lambda ARN: {lambda_arn}")
    
    print("\n" + "=" * 80)
    print("✓ SQS INFRASTRUCTURE SETUP COMPLETE!")
    print("=" * 80)
    print(f"\nResources Created:")
    print(f"  • SQS Queue URL: {queue_url}")
    print(f"  • SQS Queue ARN: {queue_arn}")
    print(f"  • Lambda Function: fraud-inference-log-consumer")
    print(f"  • Event Source Mapping: SQS → Lambda")
    print(f"  • .env updated with SQS_QUEUE_URL")
    
    print(f"\n📊 How Inference Logging Works:")
    print(f"  1. Endpoint sends predictions to SQS (using SQS_QUEUE_URL from .env)")
    print(f"  2. Lambda consumes SQS messages and writes to Athena")
    print(f"  3. Query results in section 10 of this notebook")
    
    print(f"\n⚠️  IMPORTANT: The pipeline must be re-created AFTER this cell runs")
    print(f"  so the endpoint gets the correct SQS_QUEUE_URL baked into its environment.")
    
except ClientError as e:
    error_code = e.response['Error']['Code']
    error_msg = e.response['Error']['Message']
    
    print(f"\n✗ Setup failed: {error_code}")
    print(f"  {error_msg}")
    
    if 'AccessDenied' in error_code:
        print("\n⚠️ Still missing permissions. Check that you added the policy correctly:")
        print("  1. IAM Console → Roles → Your SageMaker role")
        print("  2. Check 'Permissions' tab for 'FraudDetectionPipelinePermissions'")
        print("  3. Verify it includes SQS and Lambda actions")
    
except FileNotFoundError as e:
    print(f"\n✗ File not found: {e}")
    print("\nThe lambda_inference_logger.py file should be at src/drift_monitoring/.")
    print(f"Expected location: {project_root}/src/drift_monitoring/lambda_inference_logger.py")
    
except Exception as e:
    print(f"\n✗ Setup failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)

✓ Loaded environment from: /Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/.env
Setting up SQS Lambda Infrastructure for Inference Logging
✓ Using role from .env

Configuration:
  Queue Name: fraud-inference-logs
  Region: us-east-1
  Role: arn:aws:iam::<ACCOUNT_ID>:role/service-role/AmazonSageMaker-ExecutionRole-20250722T131288

Step 1: Creating SQS Queue...
✓ SQS Queue created: https://sqs.us-east-1.amazonaws.com/<ACCOUNT_ID>/fraud-inference-logs

Step 2: Updating .env with SQS_QUEUE_URL...
✓ .env updated with SQS_QUEUE_URL=https://sqs.us-east-1.amazonaws.com/<ACCOUNT_ID>/fraud-inference-logs

Step 3: Creating Lambda function...
✓ Lambda consumer created
  Lambda ARN: arn:aws:lambda:us-east-1:<ACCOUNT_ID>:function:fraud-inference-log-consumer

✓ SQS INFRASTRUCTURE SETUP COMPLETE!

Resources Created:
  • SQS Queue URL: https://sqs.us-east-1.amazonaws.com/<ACCOUNT_ID>/fraud-inference-logs
  • SQS Queue ARN: arn:aws:sqs:us-east-1:<ACCOU

In [6]:
import os
import sys
import boto3
from sagemaker.core.helper.session_helper import Session
from pathlib import Path
from dotenv import load_dotenv
from src.utils.aws_utils import get_execution_role

# Determine project root and load .env
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir
env_path = project_root / '.env'

# Load .env file with override=True to ensure values are loaded
if env_path.exists():
    load_dotenv(env_path, override=True)
    print(f"✓ Loaded environment from: {env_path}\n")
else:
    print(f"⚠ .env file not found at: {env_path}\n")

# Initialize AWS clients
region = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
sagemaker_client = boto3.client('sagemaker', region_name=region)
s3_client = boto3.client('s3', region_name=region)

# Get SageMaker session and role using project's wrapper
# This handles fallback to .env and provides better error messages
role = get_execution_role()

sagemaker_session = Session()
default_bucket = sagemaker_session.default_bucket()

# Display configuration
print(f"AWS Configuration:")
print(f"  Region: {region}")
print(f"  Role: {role}")
print(f"  Default S3 bucket: {default_bucket}")

# MLflow Tracking Configuration
mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
mlflow_experiment = os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection')

print(f"\nMLflow Configuration:")
if mlflow_uri:
    print(f"  Tracking URI: {mlflow_uri}")
    print(f"  Experiment: {mlflow_experiment}")
    print(f"\n✓ MLflow tracking is ENABLED")
    print(f"  Training metrics will be logged to MLflow")
    print(f"  View experiments at: SageMaker Console → MLflow")
else:
    print(f"  Tracking URI: Not set in .env")
    print(f"\n⚠ MLflow tracking is NOT configured")
    print(f"  Add MLFLOW_TRACKING_URI to .env file to enable tracking")
    print(f"  Example: MLFLOW_TRACKING_URI=arn:aws:sagemaker:us-east-1:123456789012:mlflow-app/app-XXXXX")

✓ Loaded environment from: /Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/.env

Using SageMaker execution role from environment: arn:aws:iam::<ACCOUNT_ID>:role/service-role/AmazonSageMaker-ExecutionRole-20250722T131288
sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/skoppar/Library/Application Support/sagemaker/config.yaml
AWS Configuration:
  Region: us-east-1
  Role: arn:aws:iam::<ACCOUNT_ID>:role/service-role/AmazonSageMaker-ExecutionRole-20250722T131288
  Default S3 bucket: sagemaker-us-east-1-<ACCOUNT_ID>

MLflow Configuration:
  Tracking URI: arn:aws:sagemaker:us-east-1:<ACCOUNT_ID>:mlflow-app/app-VG5VVXAMF5GQ
  Experiment: credit-card-fraud-detection-training

✓ MLflow tracking is ENABLED
  Training metrics will be logged to MLflow
  View experiments at: SageMaker Console → MLflow


### 📊 Scalability Architecture (Proof of Concept)

This pipeline demonstrates enterprise-scale ML capabilities:

**PySpark Preprocessing (Distributed)**
- **Current:** 1x ml.m5.xlarge instance with Spark processing 284K rows
- **Why 1 instance:** Avoids YARN multi-node coordination overhead for smaller datasets
- **Scales to:** 10M+ rows by increasing `instance_count` to 2-16 instances
- **Benefits:** Can handle billions of rows with proper cluster configuration
- **Data Source:** Athena via AWS Glue Data Catalog (industry standard for lakehouse)

**Key Point:** PySpark demonstrates production-ready distributed processing architecture even on 1 node. The same code scales horizontally to handle enterprise data volumes.

**XGBoost Training (Distributed)**  
- **Current:** 1x ml.m5.xlarge instance
- **Scales to:** Multi-instance distributed training
  - Set `instance_count > 1` for horizontal scaling
  - XGBoost automatically distributes training across instances
  - Example: 4x ml.m5.2xlarge for 10M+ rows
- **Framework:** SageMaker's built-in XGBoost with distributed training support

**Serverless Inference**
- **Current:** Serverless endpoint with auto-scaling
- **Scales to:** Handles traffic spikes automatically (1-200 concurrent requests)
- **Cost:** Pay only for inference time, no idle costs

**PoC Demonstrates:**
- ✅ PySpark for distributed data processing
- ✅ Athena integration via Glue Data Catalog
- ✅ Scalable training (XGBoost distributed)
- ✅ Auto-scaling inference (Serverless)
- ✅ MLflow tracking at scale
- ✅ Production-ready architecture

**To scale up:** 
- Preprocessing: Edit `pipeline.py` line 289: `instance_count=4` (or more)
- Training: Set `instance_count > 1` in pipeline parameters

In [7]:
# Pipeline configuration
PIPELINE_NAME = "fraud-detection-pipeline"
PIPELINE_DESCRIPTION = "Fraud detection ML pipeline with training, evaluation, and deployment"

# Pipeline parameters (optional - these can be overridden during execution)
PIPELINE_PARAMS = {
    # Data parameters
    #'InputDataPath': f"s3://{os.getenv('DATA_S3_BUCKET', default_bucket)}/{os.getenv('DATA_S3_PREFIX', 'fraud-detection/')}", Switching to use Athena as data source instead
    'AthenaTable' : 'training_data',
    #'AthenaFilter': "",
    # Training parameters
    'MaxDepth': '8',
    'LearningRate': '0.05',
    'NumBoostRound': '200',
    #'MinChildWeight': '1', -- removed 
    #'Subsample': '0.8', -- removed
    #'ColsampleByTree': '0.8', -- removed
    
    # Quality gates
    'MinRocAuc': '0.70',
    'MinPrAuc': '0.30',
    
    # Deployment parameters
    'EndpointName': 'fraud-detector-endpoint',
    #'ServerlessMemorySize': '2048',
    #'ServerlessMaxConcurrency': '10',
    'EndpointMemorySize': '2048',
    'EndpointMaxConcurrency': '10',
    'EnableAthenaLogging': 'true',
    'TestNumSamples':'50',
    'ModelApprovalStatus':'Approved',
    'ModelPackageGroup': 'fraud-detection'
    
    
}

print("Pipeline Configuration:")
print(f"  Name: {PIPELINE_NAME}")
print(f"  Description: {PIPELINE_DESCRIPTION}")
print(f"\nDefault Parameters:")
for key, value in PIPELINE_PARAMS.items():
    print(f"  {key}: {value}")

Pipeline Configuration:
  Name: fraud-detection-pipeline
  Description: Fraud detection ML pipeline with training, evaluation, and deployment

Default Parameters:
  AthenaTable: training_data
  MaxDepth: 8
  LearningRate: 0.05
  NumBoostRound: 200
  MinRocAuc: 0.70
  MinPrAuc: 0.30
  EndpointName: fraud-detector-endpoint
  EndpointMemorySize: 2048
  EndpointMaxConcurrency: 10
  EnableAthenaLogging: true
  TestNumSamples: 50
  ModelApprovalStatus: Approved
  ModelPackageGroup: fraud-detection


## 3. Create/Update Pipeline

This will create the pipeline definition in SageMaker using the fixed `train.py` code.

In [8]:
# 1. Delete the old pipeline. If you see updates are not getting refreshed, this will refresh the code
try:
    sagemaker_client.delete_pipeline(PipelineName=PIPELINE_NAME)
    print("✓ Pipeline deleted")
except sagemaker_client.exceptions.ResourceNotFound:
    print("ℹ Pipeline does not exist (already deleted or never created)")
except Exception as e:
    print(f"⚠ Error deleting pipeline: {e}")

ℹ Pipeline does not exist (already deleted or never created)


In [9]:
from src.train_pipeline.pipeline import create_fraud_detection_pipeline

print("Creating/updating pipeline...\n")

# Display MLflow integration info
mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
if mlflow_uri:
    print("MLflow Integration:")
    print(f"  ✓ Training step will log to: {mlflow_uri}")
    print(f"  ✓ Experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
    print(f"  ✓ Model registry: {os.getenv('MLFLOW_MODEL_NAME', 'xgboost-fraud-detector')}")
    print()

try:
    # Create pipeline builder
    pipeline_builder = create_fraud_detection_pipeline(
        pipeline_name=PIPELINE_NAME,
        region=region,
        role=role
    )
    
    # Upsert pipeline (create if doesn't exist, update if it does)
    result = pipeline_builder.upsert_pipeline(
        description=PIPELINE_DESCRIPTION,
        include_deployment=True,  # Set to False to skip deployment steps
        tags=[
            {'Key': 'Project', 'Value': 'FraudDetection'},
            {'Key': 'Environment', 'Value': 'Production'},
            {'Key': 'ManagedBy', 'Value': 'MLflow'},
        ]
    )
    
    print("✓ Pipeline created/updated successfully!")
    print(f"\nPipeline ARN: {result['pipeline_arn']}")
    
except Exception as e:
    print(f"✗ Failed to create pipeline: {e}")
    import traceback
    traceback.print_exc()

[04/16/26 08:50:41] INFO     Loaded environment from:                                                ]8;id=9006857;file:///Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/pipeline.py\pipeline.py]8;;\:]8;id=9006858;file:///Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/pipeline.py#92\92]8;;\
                             /Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated               
                             -drift-and-trend-monitoring/.env                                                      

                    INFO     MLflow tracking URI:                                                   ]8;id=9006864;file:///Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/pipeline.py\pipeline.py]8;;\:]8;id=9006865;file:///Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/pipeline.py#115\115]8;;\
                             arn:aws:sagemaker:us-east-1:<ACCOUNT_ID>:mlflow-app/app-VG5VVXAMF5GQ                  

Creating/updating pipeline...

MLflow Integration:
  ✓ Training step will log to: arn:aws:sagemaker:us-east-1:<ACCOUNT_ID>:mlflow-app/app-VG5VVXAMF5GQ
  ✓ Experiment: credit-card-fraud-detection-training
  ✓ Model registry: xgboost-fraud-detector



[04/16/26 08:50:42] INFO     Loading cached SSO token for default                                     ]8;id=9006872;file:///Users/skoppar/workspace/sample-mlops-bestpractices/.venv/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=9006873;file:///Users/skoppar/workspace/sample-mlops-bestpractices/.venv/lib/python3.12/site-packages/botocore/tokens.py#312\312]8;;\

[04/16/26 08:50:44] INFO     Initialized pipeline: fraud-detection-pipeline                         ]8;id=9006879;file:///Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/pipeline.py\pipeline.py]8;;\:]8;id=9006880;file:///Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/pipeline.py#191\191]8;;\

...

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=9007330;file:///Users/skoppar/workspace/sample-mlops-bestpractices/.venv/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=9007331;file:///Users/skoppar/workspace/sample-mlops-bestpractices/.venv/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=9007336;file:///Users/skoppar/workspace/sample-mlops-bestpractices/.venv/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=9007337;file:///Users/skoppar/workspace/sample-mlops-bestpractices/.venv/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelName' from the pipeline definition by default since ]8;id=9007342;file:///Users/skoppar/workspace/sample-mlops-bestpractices/.venv/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=9007343;file:///Users/skoppar/workspace/sample-mlops-bestpractices/.venv/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             it will be overridden at pipeline execution time. Please utilize the                  
                             PipelineDefinitionConfig to persist this field in the pipeline                        
                             definition if desired.                                                                

[04/16/26 08:51:00] INFO     ✓ Pipeline upserted:                                                  ]8;id=9007349;file:///Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/pipeline.py\pipeline.py]8;;\:]8;id=9007350;file:///Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/pipeline.py#1027\1027]8;;\
                             arn:aws:sagemaker:us-east-1:<ACCOUNT_ID>:pipeline/fraud-detection-pip                 
                             eline                                                                                 

✓ Pipeline created/updated successfully!

Pipeline ARN: arn:aws:sagemaker:us-east-1:<ACCOUNT_ID>:pipeline/fraud-detection-pipeline


### 🔧 Quick Fix: Add Glue Data Catalog Permissions

The most common PySpark failure is **missing Glue Data Catalog permissions**.

PySpark reads from Athena via the Glue Data Catalog (Hive metastore). Your SageMaker role needs:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "GlueDataCatalogAccess",
      "Effect": "Allow",
      "Action": [
        "glue:GetDatabase",
        "glue:GetDatabases", 
        "glue:GetTable",
        "glue:GetTables",
        "glue:GetPartition",
        "glue:GetPartitions",
        "glue:BatchGetPartition"
      ],
      "Resource": "*"
    },
    {
      "Sid": "LakeFormationDataAccess",
      "Effect": "Allow",
      "Action": [
        "lakeformation:GetDataAccess"
      ],
      "Resource": "*"
    }
  ]
}
```

**How to add:**
1. Go to IAM Console → Roles → Your SageMaker role
2. Add inline policy → paste JSON above
3. Name it: `GlueDataCatalogAccess`
4. Save and re-run the pipeline

**Your role:** `arn:aws:iam::YOUR_ACCOUNT_ID:role/service-role/AmazonSageMaker-ExecutionRole-20250722T131288`

## 4. Start Pipeline Execution

This will start a new execution of the pipeline with the fixed training code.

### Common PySpark Preprocessing Errors

**1. Glue/Athena Permissions Error**
```
AccessDeniedException: User is not authorized to perform: glue:GetTable
```
**Fix:** Add Glue Data Catalog permissions to SageMaker role:
```json
{
  "Effect": "Allow",
  "Action": [
    "glue:GetDatabase",
    "glue:GetTable",
    "glue:GetPartitions",
    "lakeformation:GetDataAccess"
  ],
  "Resource": "*"
}
```

**2. Table Not Found**
```
AnalysisException: Table or view not found: fraud_detection.training_data
```
**Fix:** Check that Athena table exists and database name is correct in .env:
- `ATHENA_DATABASE=fraud_detection`
- Verify table: `SELECT * FROM fraud_detection.training_data LIMIT 1`

**3. PySpark Module Not Found**
```
ModuleNotFoundError: No module named 'pyspark'
```
**Fix:** This shouldn't happen with PySparkProcessor, but if it does, check instance type supports Spark.

**4. Memory/Resource Issues**
```
ExecutorLostFailure: Worker lost
```
**Fix:** Increase instance size or count in pipeline config (currently using 2x ml.m5.xlarge)

**5. YARN Connection Error**
```
spark-submit --master yarn failed
```
**Fix:** YARN mode requires EMR or properly configured cluster. For SageMaker, ensure PySparkProcessor is used correctly.

In [10]:
# Generate execution name with timestamp
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
execution_name = f"{PIPELINE_NAME}-{timestamp}"

print(f"Starting pipeline execution: {execution_name}\n")

# Format parameters for SageMaker
pipeline_parameters = [
    {'Name': key, 'Value': str(value)}
    for key, value in PIPELINE_PARAMS.items()
]

try:
    response = sagemaker_client.start_pipeline_execution(
        PipelineName=PIPELINE_NAME,
        PipelineExecutionDisplayName=execution_name,
        PipelineParameters=pipeline_parameters
    )
    
    execution_arn = response['PipelineExecutionArn']
    
    print("✓ Pipeline execution started successfully!")
    print(f"\nExecution Details:")
    print(f"  ARN: {execution_arn}")
    print(f"  Name: {execution_name}")
    
    # MLflow logging information
    mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
    if mlflow_uri:
        print(f"\nMLflow Tracking:")
        print(f"  📊 Training metrics are being logged to MLflow")
        print(f"  📈 Tracking Server: {mlflow_uri}")
        print(f"  🔬 Experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
        print(f"  📦 Model Registry: {os.getenv('MLFLOW_MODEL_NAME', 'xgboost-fraud-detector')}")
        print(f"\n  View in MLflow UI:")
        print(f"    1. Open SageMaker Console → MLFlow")
        print(f"    2. Select your MLflow app")
        print(f"    3. Find experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
    else:
        print(f"\n⚠ MLflow tracking not configured")
    
    print(f"\nYou can monitor this execution in the next cell")
    
    # Store for monitoring
    CURRENT_EXECUTION_ARN = execution_arn
    
except Exception as e:
    print(f"✗ Failed to start execution: {e}")
    import traceback
    traceback.print_exc()

Starting pipeline execution: fraud-detection-pipeline-20260416-085100

✓ Pipeline execution started successfully!

Execution Details:
  ARN: arn:aws:sagemaker:us-east-1:<ACCOUNT_ID>:pipeline/fraud-detection-pipeline/execution/ilgkw673e15r
  Name: fraud-detection-pipeline-20260416-085100

MLflow Tracking:
  📊 Training metrics are being logged to MLflow
  📈 Tracking Server: arn:aws:sagemaker:us-east-1:<ACCOUNT_ID>:mlflow-app/app-VG5VVXAMF5GQ
  🔬 Experiment: credit-card-fraud-detection-training
  📦 Model Registry: xgboost-fraud-detector

  View in MLflow UI:
    1. Open SageMaker Console → MLFlow
    2. Select your MLflow app
    3. Find experiment: credit-card-fraud-detection-training

You can monitor this execution in the next cell


## 5. Monitor Pipeline Execution

Watch the pipeline execution in real-time.

In [11]:
# Generate execution name with timestamp
# from datetime import datetime
# timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
# execution_name = f"{PIPELINE_NAME}-{timestamp}"

# print(f"Starting pipeline execution: {execution_name}\n")

# # Format parameters for SageMaker
# pipeline_parameters = [
#     {'Name': key, 'Value': str(value)}
#     for key, value in PIPELINE_PARAMS.items()
# ]

# try:
#     response = sagemaker_client.start_pipeline_execution(
#         PipelineName=PIPELINE_NAME,
#         PipelineExecutionDisplayName=execution_name,
#         PipelineParameters=pipeline_parameters
#     )
    
#     execution_arn = response['PipelineExecutionArn']
    
#     print("✓ Pipeline execution started successfully!")
#     print(f"\nExecution Details:")
#     print(f"  ARN: {execution_arn}")
#     print(f"  Name: {execution_name}")
    
#     # MLflow logging information
#     mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
#     if mlflow_uri:
#         print(f"\nMLflow Tracking:")
#         print(f"  📊 Training metrics are being logged to MLflow")
#         print(f"  📈 Tracking Server: {mlflow_uri}")
#         print(f"  🔬 Experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
#         print(f"  📦 Model Registry: {os.getenv('MLFLOW_MODEL_NAME', 'xgboost-fraud-detector')}")
#         print(f"\n  View in MLflow UI:")
#         print(f"    1. Open SageMaker Console → MLFlow")
#         print(f"    2. Select your MLflow app")
#         print(f"    3. Find experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
#     else:
#         print(f"\n⚠ MLflow tracking not configured")
    
#     print(f"\nYou can monitor this execution in the next cell")
    
#     # Store for monitoring
#     CURRENT_EXECUTION_ARN = execution_arn
    
# except Exception as e:
#     print(f"✗ Failed to start execution: {e}")
#     import traceback
#     traceback.print_exc()

In [12]:
  # Check actual model performance from evaluation report                                                                                                                                                                                                         
def get_actual_metrics(execution_arn):                                                                                                                                                                                                                          
      """Get actual metrics from the evaluation report."""                                                                                                                                                                                                        
      import boto3
      import json

      sagemaker_client = boto3.client('sagemaker', region_name=AWS_DEFAULT_REGION)
      s3_client = boto3.client('s3', region_name=AWS_DEFAULT_REGION)

      # Get execution steps
      response = sagemaker_client.list_pipeline_execution_steps(
          PipelineExecutionArn=execution_arn
      )

      steps = response.get('PipelineExecutionSteps', [])

      # Find evaluation step
      for step in steps:
          if 'EvaluateModel' in step['StepName']:
              if 'Metadata' in step and 'ProcessingJob' in step['Metadata']:
                  job_name = step['Metadata']['ProcessingJob']['Arn'].split('/')[-1]

                  # Get processing job details
                  response = sagemaker_client.describe_processing_job(
                      ProcessingJobName=job_name
                  )

                  # Find evaluation output
                  for output in response['ProcessingOutputConfig']['Outputs']:
                      if output['OutputName'] == 'evaluation':
                          s3_uri = output['S3Output']['S3Uri']

                          # Parse S3 URI
                          parts = s3_uri.replace('s3://', '').split('/', 1)
                          bucket = parts[0]
                          key = parts[1] + '/evaluation.json'

                          print(f"Fetching evaluation report from:")
                          print(f"  s3://{bucket}/{key}\n")

                          try:
                              # Download evaluation report
                              obj = s3_client.get_object(Bucket=bucket, Key=key)
                              property_data = json.loads(obj['Body'].read())

                              print("="*80)
                              print("ACTUAL MODEL PERFORMANCE:")
                              print("="*80)

                              metrics = property_data.get('binary_classification_metrics', {})
                              for metric_name, metric_value in metrics.items():
                                  value = metric_value.get('value', 'N/A')
                                  print(f"  {metric_name.upper()}: {value:.4f}" if isinstance(value, float) else f"  {metric_name.upper()}: {value}")

                              print("="*80)

                              # Also check full report
                              key_full = parts[1] + '/evaluation_report.json'
                              try:
                                  obj_full = s3_client.get_object(Bucket=bucket, Key=key_full)
                                  full_report = json.loads(obj_full['Body'].read())

                                  print("\nQUALITY GATES:")
                                  print("="*80)
                                  qg = full_report.get('quality_gates', {})
                                  print(f"  Passed: {qg.get('passed', 'N/A')}")

                                  for check in qg.get('checks', []):
                                      status = "✓" if check['passed'] else "✗"
                                      print(f"  {status} {check['metric'].upper()}: {check['value']:.4f} (threshold: {check['threshold']:.4f})")

                                  if qg.get('failures'):
                                      print(f"\n  Failures: {', '.join(qg['failures'])}")

                                  print("="*80)

                              except Exception as e:
                                  print(f"Could not read full report: {e}")

                              return property_data

                          except Exception as e:
                              print(f"✗ Could not read evaluation report: {e}")

  # Run for current execution
if 'CURRENT_EXECUTION_ARN' in locals():
      get_actual_metrics(CURRENT_EXECUTION_ARN)
else:
      print("No execution found. Set CURRENT_EXECUTION_ARN first.")

## 6. Utility Functions

Additional helper functions for pipeline management.

In [13]:
def stop_execution(execution_arn):
    """Stop a running pipeline execution."""
    try:
        response = sagemaker_client.stop_pipeline_execution(
            PipelineExecutionArn=execution_arn
        )
        print(f"✓ Stopped execution: {execution_arn}")
        return response
    except Exception as e:
        print(f"✗ Failed to stop execution: {e}")
        return None

def delete_pipeline(pipeline_name):
    """Delete a pipeline (use with caution!)."""
    try:
        response = sagemaker_client.delete_pipeline(
            PipelineName=pipeline_name
        )
        print(f"✓ Deleted pipeline: {pipeline_name}")
        return response
    except sagemaker_client.exceptions.ResourceNotFound:
        print(f"ℹ Pipeline '{pipeline_name}' does not exist (already deleted or never created)")
        return None
    except Exception as e:
        print(f"✗ Failed to delete pipeline: {e}")
        return None

def get_pipeline_definition(pipeline_name):
    """Get pipeline definition JSON."""
    try:
        response = sagemaker_client.describe_pipeline(
            PipelineName=pipeline_name
        )
        definition = json.loads(response['PipelineDefinition'])
        return definition
    except Exception as e:
        print(f"✗ Failed to get pipeline definition: {e}")
        return None

print("Utility functions loaded:")
print("  - stop_execution(execution_arn)")
print("  - delete_pipeline(pipeline_name)")
print("  - get_pipeline_definition(pipeline_name)")

Utility functions loaded:
  - stop_execution(execution_arn)
  - delete_pipeline(pipeline_name)
  - get_pipeline_definition(pipeline_name)


In [14]:
import boto3, pandas as pd, io, time

athena_client = boto3.client('athena', region_name=region)
s3_client = boto3.client('s3', region_name=region)

response = athena_client.start_query_execution(
    QueryString="SELECT * FROM training_data LIMIT 3",
    QueryExecutionContext={'Database': 'fraud_detection'},
    ResultConfiguration={'OutputLocation': f's3://{default_bucket}/athena-query-results/'}
)

time.sleep(10)

response = athena_client.get_query_execution(QueryExecutionId=response['QueryExecutionId'])
output_uri = response['QueryExecution']['ResultConfiguration']['OutputLocation']
bucket = output_uri.replace('s3://', '').split('/')[0]
key = '/'.join(output_uri.replace('s3://', '').split('/')[1:])

obj = s3_client.get_object(Bucket=bucket, Key=key)
df = pd.read_csv(io.BytesIO(obj['Body'].read()))

print(f"Athena columns: {list(df.columns)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nFirst 2 rows:")
print(df.head(2))

Athena columns: ['transaction_id', 'transaction_timestamp', 'transaction_hour', 'transaction_day_of_week', 'transaction_amount', 'transaction_type_code', 'customer_age', 'customer_gender', 'customer_tenure_months', 'account_age_days', 'distance_from_home_km', 'distance_from_last_transaction_km', 'time_since_last_transaction_min', 'online_transaction', 'international_transaction', 'high_risk_country', 'merchant_category_code', 'merchant_reputation_score', 'chip_transaction', 'pin_used', 'card_present', 'cvv_match', 'address_verification_match', 'num_transactions_24h', 'num_transactions_7days', 'avg_transaction_amount_30days', 'max_transaction_amount_30days', 'velocity_score', 'recurring_transaction', 'previous_fraud_incidents', 'credit_limit', 'available_credit_ratio', 'fraud_prediction', 'fraud_probability', 'is_fraud', 'data_version', 'created_at', 'updated_at']
Total columns: 38

First 2 rows:
   transaction_id  transaction_timestamp  transaction_hour  \
0          280000            

## 7. Quick Reference

### Common Tasks

**Start a new execution with custom parameters:**
```python
PIPELINE_PARAMS['MinRocAuc'] = '0.90'
PIPELINE_PARAMS['EndpointName'] = 'fraud-detector-v2'
# Then run cell 4 to start execution
```

**Stop a running execution:**
```python
stop_execution(CURRENT_EXECUTION_ARN)
```

**Monitor a specific execution:**
```python
execution_arn = 'arn:aws:sagemaker:us-east-1:123456789012:pipeline/fraud-detection-pipeline/execution/xyz'
monitor_execution(execution_arn)
```

**View CloudWatch logs:**
- Go to CloudWatch Console
- Navigate to Log Groups
- Search for `/aws/sagemaker/ProcessingJobs` or `/aws/sagemaker/TrainingJobs`

### Expected Results

With `train.py`, you should see:
- ✓ **ROC-AUC > 0.85** (likely 0.90-0.95)
- ✓ **PR-AUC > 0.50** (likely 0.60-0.80)
- ✓ **True Positives > 0** (model detects fraud)
- ✓ **Quality gates pass**
- ✓ **Pipeline completes successfully**


### Troubleshooting

**Pipeline creation fails:**
- Check that `.env` file is properly configured
- Verify SageMaker execution role has necessary permissions
- Check S3 bucket access

**Training step fails:**
- Check CloudWatch logs for the training job
- Verify input data path is correct
- Check that preprocessing completed successfully

**Evaluation fails quality gates:**
- Check evaluation metrics in cell 6
- If ROC-AUC is still 0.50, verify the pipeline picked up the fixed code
- Consider adjusting quality gate thresholds in PIPELINE_PARAMS